05_robustness_testing.ipynb - 鲁棒性 & 对抗测试

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

In [ ]:
# ── 1. 加载数据 & 模型 ─────────────────────────────────────
df = pd.read_csv('../data/processed/train_feat.csv')
X  = df.drop(columns=['Survived'])
y  = df['Survived']

_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = joblib.load('../models/randomforest.joblib')
print(f"测试集维度: {X_test.shape}")
print(f"列名: {X_test.columns.tolist()}")

In [ ]:
# ── 2. 基准性能 ────────────────────────────────────────────
def evaluate(model, X, y, label='baseline'):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    acc    = accuracy_score(y, y_pred)
    auc    = roc_auc_score(y, y_prob)
    print(f"[{label:35s}]  acc={acc:.4f}  auc={auc:.4f}")
    return {'label': label, 'acc': acc, 'auc': auc}

results = []
results.append(evaluate(model, X_test, y_test, 'baseline'))

In [ ]:
# ── 3. Perturbation 函数 ───────────────────────────────────

def add_gaussian_noise(X, cols, noise_std=0.5):
    """对数值列加高斯噪声"""
    X_p = X.copy()
    for col in cols:
        if col in X_p.columns:
            X_p[col] += np.random.normal(0, noise_std, size=len(X_p))
    return X_p

def random_mislabel(X, cols, flip_ratio=0.2):
    """对 0/1 类别列随机翻转"""
    X_p = X.copy()
    for col in cols:
        if col in X_p.columns:
            mask = np.random.rand(len(X_p)) < flip_ratio
            X_p.loc[mask, col] = 1 - X_p.loc[mask, col]
    return X_p

def extreme_values(X, col_val_map):
    """将指定列替换为极端值"""
    X_p = X.copy()
    for col, val in col_val_map.items():
        if col in X_p.columns:
            X_p[col] = val
    return X_p


In [ ]:
# ── 4. 执行各项干扰测试 ────────────────────────────────────
np.random.seed(42)

# 4-1 Age 加噪声（std=0.5）
X_p = add_gaussian_noise(X_test, ['Age'], noise_std=0.5)
results.append(evaluate(model, X_p, y_test, 'gaussian_noise_age_std0.5'))

# 4-2 Fare 加噪声（std=0.5）
X_p = add_gaussian_noise(X_test, ['Fare'], noise_std=0.5)
results.append(evaluate(model, X_p, y_test, 'gaussian_noise_fare_std0.5'))

# 4-3 Age + Fare 同时加噪声
X_p = add_gaussian_noise(X_test, ['Age', 'Fare'], noise_std=0.5)
results.append(evaluate(model, X_p, y_test, 'gaussian_noise_age+fare_std0.5'))

# 4-4 强噪声（std=2.0）
X_p = add_gaussian_noise(X_test, ['Age', 'Fare'], noise_std=2.0)
results.append(evaluate(model, X_p, y_test, 'gaussian_noise_age+fare_std2.0'))

# 4-5 Sex_male 随机翻转 20%
X_p = random_mislabel(X_test, ['Sex_male'], flip_ratio=0.2)
results.append(evaluate(model, X_p, y_test, 'mislabel_sex_20pct'))

# 4-6 Embarked 随机翻转 20%
X_p = random_mislabel(X_test, ['Embarked_Q', 'Embarked_S'], flip_ratio=0.2)
results.append(evaluate(model, X_p, y_test, 'mislabel_embarked_20pct'))

# 4-7 极端值：Age=0
X_p = extreme_values(X_test, {'Age': 0})
results.append(evaluate(model, X_p, y_test, 'extreme_age=0'))

# 4-8 极端值：Fare=0
X_p = extreme_values(X_test, {'Fare': 0})
results.append(evaluate(model, X_p, y_test, 'extreme_fare=0'))

# 4-9 极端值：Age=0 且 Fare=0
X_p = extreme_values(X_test, {'Age': 0, 'Fare': 0})
results.append(evaluate(model, X_p, y_test, 'extreme_age=0_fare=0'))

In [ ]:
# ── 5. 汇总结果 ────────────────────────────────────────────
df_results = pd.DataFrame(results)
print("\n── 汇总 ──")
print(df_results.to_string(index=False))

baseline_acc = df_results.loc[0, 'acc']
baseline_auc = df_results.loc[0, 'auc']
df_results['acc_drop'] = baseline_acc - df_results['acc']
df_results['auc_drop'] = baseline_auc - df_results['auc']

In [ ]:
# ── 6. 断言测试 ────────────────────────────────────────────
print("\n── 断言测试 ──")

# 弱噪声下 accuracy 下降不超过 10%
weak_noise = df_results[df_results['label'] == 'gaussian_noise_age+fare_std0.5'].iloc[0]
assert weak_noise['acc_drop'] < 0.10, \
    f"[FAIL] 弱噪声 acc 下降过大: {weak_noise['acc_drop']:.4f}"
print(f"[PASS] 弱噪声 acc 下降在可接受范围: {weak_noise['acc_drop']:.4f} < 0.10")

# AUC 始终大于 0.5（优于随机）
for _, row in df_results.iterrows():
    assert row['auc'] > 0.5, \
        f"[FAIL] {row['label']} AUC <= 0.5: {row['auc']:.4f}"
print("[PASS] 所有场景 AUC > 0.5")

In [ ]:
# ── 7. 可视化 & 保存 ───────────────────────────────────────
os.makedirs('../results', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels_short = [l.replace('gaussian_noise_', 'noise_')
                  .replace('mislabel_', 'mis_')
                  .replace('extreme_', 'ext_')
                for l in df_results['label']]

axes[0].barh(labels_short, df_results['acc'], color='steelblue')
axes[0].axvline(baseline_acc, color='red', linestyle='--', label='baseline')
axes[0].set_title('Accuracy under Perturbations')
axes[0].set_xlabel('Accuracy')
axes[0].legend()

axes[1].barh(labels_short, df_results['auc'], color='darkorange')
axes[1].axvline(baseline_auc, color='red', linestyle='--', label='baseline')
axes[1].set_title('AUC under Perturbations')
axes[1].set_xlabel('AUC')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/robustness_plot.png', dpi=150)
plt.show()
print("[PASS] 图表已保存: results/robustness_plot.png")

In [ ]:
# ── 8. 写入 robustness.txt ─────────────────────────────────
with open('../results/robustness.txt', 'w', encoding='utf-8') as f:
    f.write("Robustness Test Results\n")
    f.write("=" * 60 + "\n")
    f.write(f"baseline_acc={baseline_acc:.4f}  baseline_auc={baseline_auc:.4f}\n\n")
    for _, row in df_results.iterrows():
        f.write(
            f"{row['label']:40s}  "
            f"acc={row['acc']:.4f}  "
            f"auc={row['auc']:.4f}  "
            f"acc_drop={row['acc_drop']:+.4f}  "
            f"auc_drop={row['auc_drop']:+.4f}\n"
        )

print("[PASS] 结果已写入: results/robustness.txt")